# MARSNet — Final Test Set Evaluation

Runs all four models on the **sealed test set** in one session.
This notebook is built directly from the R20 notebook's working inference pipeline —
same `predict_sequence` function, same data loading, same normalisation.

## Models evaluated
- MARSNet (R20 retrain — `marsnet_r20_final.pt`)
- LSTM baseline (`marsnet_lstm_best.pt`)
- EKF v2 (quaternion gravity compensation, Nelder-Mead tuned on val)
- Naive (constant-velocity from last known GPS)

## Rules
- Run val set first (Cell 12) and confirm numbers match before running test set
- Run test set **once only** (Cell 13) — these numbers go directly into Table 1
- Do not re-run after seeing results
- Do not change any parameter based on test-set numbers

In [ ]:
import math, os, time, csv, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Configuration (portable — no Colab / Google Drive required) ───────────────
# Set GATEIO_DATA / GATEIO_CKPT_DIR, or edit the defaults below.
import os
_CKPT = os.environ.get("GATEIO_CKPT_DIR", "../results/checkpoints")
DATA_PATH = os.environ.get("GATEIO_DATA", "../data/processed/MARS_Master_Dataset.npz")
CKPT_R20  = os.path.join(_CKPT, "marsnet_r20_final.pt")
CKPT_LSTM = os.path.join(_CKPT, "marsnet_lstm_best.pt")
OUT_DIR   = os.environ.get("GATEIO_OUT", "./results_test")
VAL_DIR   = os.environ.get("GATEIO_VAL_OUT", "./results_val")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
# ── Constants — copied verbatim from R20 notebook ─────────────────────────────
D_MODEL    = 48
N_TCN_STACKS = 2
N_HEADS    = 4
DROPOUT    = 0.15
N_CHAN     = 14
N_IMU_CHAN = 10
SEQ_LEN    = 300
WIN_LEN    = 200
DT         = 0.1
HUBER_DELTA = 0.3
TURN_GYRO_THR  = 0.10
ZUPT_GYRO_THR  = 0.05
CVPRIOR_GYRO_THR = 0.10
LSTM_HIDDEN = 128
LSTM_LAYERS = 2

# EKF
GRAVITY  = 9.81
OE_LEN   = 100
Q_V_INIT = 1e-4
Q_B_INIT = 1e-6
R_INIT   = 1e-4

GROUP_MAP = {**{i:'straight-short' for i in range(0,35)},
             **{i:'straight-med'   for i in range(35,41)},
             **{i:'TURN'           for i in range(41,47)},
             **{i:'FALSE-ALARM'    for i in range(47,53)},
             **{i:'long-outage'    for i in range(53,59)}}
GROUP_ORDER = ['straight-short','straight-med','TURN','FALSE-ALARM','long-outage']

print('Config OK')

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────────
_npz   = np.load(DATA_PATH)
Xv     = _npz['X_val'].astype(np.float32)
Yv     = _npz['Y_val'].astype(np.float32)
Xt     = _npz['X_test'].astype(np.float32)
Yt     = _npz['Y_test'].astype(np.float32)
vi     = _npz['val_valid_idx']
ti     = _npz['test_valid_idx']
Xmed   = _npz['X_median'].astype(np.float32)
Xiq    = _npz['X_iqr'].astype(np.float32)
dv_iqr    = _npz['Y_iqr'].astype(np.float32)
dv_median = _npz['Y_median'].astype(np.float32)
SL     = int(_npz['seq_len'][0]) if 'seq_len' in _npz else SEQ_LEN
N_VAL  = len(vi)
N_TEST = len(ti)
print(f'Val:{N_VAL}  Test:{N_TEST}  SL:{SL}')
print(f'Y_iqr={dv_iqr}  Y_median={dv_median}')

In [ ]:
# ── Architecture — copied verbatim from R20 notebook (Cell 5) ─────────────────
class OutageStepPE(nn.Module):
    def __init__(self, d_model, max_steps=211):
        super().__init__()
        pe  = torch.zeros(max_steps, d_model)
        pos = torch.arange(max_steps).float().unsqueeze(1)
        div = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.)/d_model))
        pe[:,0::2] = torch.sin(pos*div);  pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe)
    def forward(self, tokens, outage_flag):
        flag   = (outage_flag > 0.5).long()
        cumsum = flag.cumsum(dim=1); reset = cumsum*(1-flag)
        steps  = (cumsum - reset.cummax(dim=1).values).clamp(0, self.pe.shape[0]-1)
        return tokens + self.pe[steps]

class TCNBlock(nn.Module):
    def __init__(self, d_model, dilation, kernel_size=3, dropout=0.1):
        super().__init__()
        pad=(kernel_size-1)*dilation; self.pad=pad
        self.conv1=nn.Conv1d(d_model,d_model,kernel_size,dilation=dilation,padding=0)
        self.norm1=nn.LayerNorm(d_model)
        self.conv2=nn.Conv1d(d_model,d_model,kernel_size,dilation=dilation,padding=0)
        self.norm2=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout); self.act=nn.GELU()
    def forward(self, x):
        res=x
        x=self.act(self.norm1(self.conv1(F.pad(x,(self.pad,0))).transpose(1,2)).transpose(1,2))
        x=self.drop(x)
        x=self.act(self.norm2(self.conv2(F.pad(x,(self.pad,0))).transpose(1,2)).transpose(1,2))
        return self.drop(x)+res

class TCNBackbone(nn.Module):
    DILATIONS=[1,2,4,8,16]
    def __init__(self, d_model, n_stacks=N_TCN_STACKS, kernel_size=3, dropout=DROPOUT):
        super().__init__()
        self.net=nn.Sequential(*[TCNBlock(d_model,d,kernel_size,dropout)
                                  for _ in range(n_stacks) for d in self.DILATIONS])
    def forward(self, x): return self.net(x.transpose(1,2)).transpose(1,2)

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.n_heads=n_heads; self.d_head=d_model//n_heads; self.scale=self.d_head**-0.5
        self.norm=nn.LayerNorm(d_model); self.qkv=nn.Linear(d_model,3*d_model,bias=False)
        self.proj=nn.Linear(d_model,d_model); self.drop=nn.Dropout(dropout)
        slopes=2.**(-8.*torch.arange(1,n_heads+1).float()/n_heads)
        self.register_buffer('slopes',slopes)
    def _bias(self,S,dev):
        pos=torch.arange(S,device=dev).float(); dist=(pos.unsqueeze(0)-pos.unsqueeze(1)).abs()
        return (-self.slopes.view(-1,1,1)*dist.unsqueeze(0)
                +torch.triu(torch.full((S,S),float('-inf'),device=dev),diagonal=1).unsqueeze(0))
    def forward(self, x):
        B,S,_=x.shape; res=x; h=self.norm(x)
        QKV=self.qkv(h).reshape(B,S,3,self.n_heads,self.d_head); Q,K,V=QKV.unbind(2)
        Q=Q.transpose(1,2); K=K.transpose(1,2); V=V.transpose(1,2)
        attn=torch.softmax(torch.matmul(Q,K.transpose(-2,-1))*self.scale+self._bias(S,x.device),dim=-1).nan_to_num(0.)
        return res+self.drop(self.proj(self.drop(attn).matmul(V).transpose(1,2).reshape(B,S,-1)))

class WindowEncoder(nn.Module):
    def __init__(self, in_channels=N_CHAN, d_model=D_MODEL):
        super().__init__()
        g=min(8,d_model//6)
        self.conv1=nn.Conv1d(in_channels,d_model,7,padding=3); self.norm1=nn.GroupNorm(g,d_model)
        self.conv2=nn.Conv1d(d_model,d_model,5,padding=2);     self.norm2=nn.GroupNorm(g,d_model)
        self.conv3=nn.Conv1d(d_model,d_model,3,padding=1);     self.norm3=nn.GroupNorm(g,d_model)
        self.drop=nn.Dropout(0.1); self.cls=nn.Parameter(torch.randn(1,1,d_model)*0.02)
        self.pool=nn.MultiheadAttention(d_model,num_heads=4,dropout=0.1,batch_first=True)
    def forward(self, x):
        x=F.gelu(self.norm1(self.conv1(x))); x=F.gelu(self.norm2(self.conv2(x)))
        x=F.gelu(self.norm3(self.conv3(x))); x=self.drop(x.transpose(1,2))
        out,_=self.pool(self.cls.expand(x.size(0),-1,-1),x,x); return out.squeeze(1)

class VelocityHead(nn.Module):
    def __init__(self, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        h=d_model//2
        def _b(): return nn.Sequential(nn.LayerNorm(d_model),nn.Linear(d_model,h),
                                        nn.GELU(),nn.Dropout(dropout),nn.Linear(h,3))
        self.head_aided=_b(); self.head_dr=_b(); self.v_prev_proj=nn.Linear(3,d_model)
    def forward(self, tokens, outage_flag, v_prev):
        alpha=outage_flag.unsqueeze(-1)
        return (1.-alpha)*self.head_aided(tokens)+alpha*self.head_dr(tokens+alpha*self.v_prev_proj(v_prev))

class MARSNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.window_enc=WindowEncoder(N_CHAN,D_MODEL); self.pos_enc=OutageStepPE(D_MODEL)
        self.tcn=TCNBackbone(D_MODEL,N_TCN_STACKS); self.attn=ALiBiCausalAttention(D_MODEL,N_HEADS,DROPOUT)
        self.head=VelocityHead(D_MODEL,DROPOUT)
    def forward(self, x, outage_flag, v_prev):
        B,S,W,C=x.shape
        tokens=self.window_enc(x.reshape(B*S,W,C).permute(0,2,1).contiguous()).view(B,S,-1)
        tokens=self.pos_enc(tokens,outage_flag); tokens=self.tcn(tokens); tokens=self.attn(tokens)
        return self.head(tokens,outage_flag,v_prev)

class LSTMBackbone(nn.Module):
    def __init__(self, d=D_MODEL, hidden=LSTM_HIDDEN, n_layers=LSTM_LAYERS):
        super().__init__()
        self.lstm=nn.LSTM(d,hidden,n_layers,batch_first=True,
                          dropout=0.15 if n_layers>1 else 0.)
        self.proj=nn.Linear(hidden,d)
    def forward(self, x, mask=None):
        out,_=self.lstm(x); return self.proj(out)

class MARSNetLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.window_enc=WindowEncoder(N_CHAN,D_MODEL); self.pos_enc=OutageStepPE(D_MODEL)
        self.backbone=LSTMBackbone(); self.head=VelocityHead(D_MODEL,DROPOUT)
    def forward(self, x, outage_flag, v_prev):
        B,S,W,C=x.shape
        tokens=self.window_enc(x.reshape(B*S,W,C).permute(0,2,1).contiguous()).view(B,S,-1)
        tokens=self.pos_enc(tokens,outage_flag); tokens=self.backbone(tokens)
        return self.head(tokens,outage_flag,v_prev)

# Load both learned models
model_r20  = MARSNet().to(DEVICE)
model_lstm = MARSNetLSTM().to(DEVICE)

ck_r20  = torch.load(CKPT_R20,  map_location=DEVICE, weights_only=False)
ck_lstm = torch.load(CKPT_LSTM, map_location=DEVICE, weights_only=False)
model_r20.load_state_dict(ck_r20['model']   if 'model' in ck_r20  else ck_r20)
model_lstm.load_state_dict(ck_lstm['model'] if 'model' in ck_lstm else ck_lstm)
model_r20.eval(); model_lstm.eval()

print(f'MARSNet: epoch={ck_r20.get("epoch")}  best_val={ck_r20.get("best_val"):.3f}m')
print(f'LSTM:    epoch={ck_lstm.get("epoch")}  best_val={ck_lstm.get("best_val"):.3f}m')
print(f'MARSNet params: {sum(p.numel() for p in model_r20.parameters()):,}')
print(f'LSTM params:    {sum(p.numel() for p in model_lstm.parameters()):,}')

In [ ]:
# ── predict_sequence — copied verbatim from R20 notebook (Cell 16) ────────────
# This is the WORKING inference function. Do not modify it.
# It takes x_norm_seq (manually normalised X) and y_raw_seq (raw Y_val values)
# and constructs the input exactly as the training data loader did.

@torch.no_grad()
def predict_sequence(model, x_norm_seq, y_raw_seq, outage_start, outage_end, dv_iqr, dv_median, device):
    model.eval(); S=x_norm_seq.shape[0]; W=x_norm_seq.shape[1]
    om=torch.zeros(S,dtype=torch.float32)
    if outage_start>=3: om[outage_start-2]=1/3; om[outage_start-1]=2/3
    om[outage_start:outage_end]=1.
    vr=torch.from_numpy(y_raw_seq).float(); vp=torch.zeros(S,3); last=vr[0].clone()
    for t in range(S):
        if om[t]>0.5: vp[t]=last
        else: vp[t]=vr[t]; last=vr[t].clone()
    xi=torch.from_numpy(x_norm_seq).float()[:,:,:N_IMU_CHAN]
    vpch=vp.unsqueeze(1).expand(-1,W,-1); fch=om.view(S,1,1).expand(-1,W,1)
    xf=torch.cat([xi,vpch,fch],dim=-1)
    dvp=model(xf.unsqueeze(0).to(device),om.unsqueeze(0).to(device),vp.unsqueeze(0).to(device)).squeeze(0).cpu()
    iqr=torch.from_numpy(dv_iqr).float(); med=torch.from_numpy(dv_median).float()
    pm=dvp*iqr+med; tm=vr
    def integrate(dv,s):
        pos=torch.zeros(S,2)
        for t in range(s+1,S): pos[t]=pos[t-1]+dv[t,:2]*DT
        return pos.numpy()
    pp=integrate(pm,outage_start); pt=integrate(tm,outage_start)
    return {'dv_pred_ms':pm.numpy(),'dv_true_ms':tm.numpy(),'pos_pred':pp,'pos_true':pt,
            'outage_start':outage_start,
            'drift_m':float(np.linalg.norm(pp[outage_end-1]-pt[outage_end-1]))}

# Quick verify on S0, S41, S47 — must match R20 retrain results
print('Verifying inference on known sequences...')
for si, exp in [(0, 0.29), (41, 2.55), (47, 29.08)]:
    start = int(vi[si])
    xrs = Xv[start:start+SL]; yrs = Yv[start:start+SL]
    xns = (xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq)
    os_ = SL//3; oe_ = min(os_+100,SL)
    r   = predict_sequence(model_r20, xns, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
    ok  = '✓' if abs(r['drift_m']-exp)<exp*0.1 else '~' if abs(r['drift_m']-exp)<exp*0.3 else '✗'
    print(f'  S{si}: {r["drift_m"]:.3f}m  (expect {exp}m from retrain)  {ok}')
print('If all ✓ or ~: proceed. If ✗: session reset — re-run cells 1-5 then this cell.')

In [ ]:
# ── EKF v2: quaternion gravity compensation ───────────────────────────────────
# Convention confirmed: scalar-first [w,x,y,z], body->nav, gravity = +9.81 on z-axis

GRAVITY_VEC = np.array([0., 0., GRAVITY], dtype=np.float64)

def quat_to_rotmat(q):
    w,x,y,z = q
    return np.array([
        [1-2*(y*y+z*z),  2*(x*y-z*w),    2*(x*z+y*w)],
        [2*(x*y+z*w),    1-2*(x*x+z*z),  2*(y*z-x*w)],
        [2*(x*z-y*w),    2*(y*z+x*w),    1-2*(x*x+y*y)]
    ], dtype=np.float64)

def run_kf_sequence(seq_idx, Q_v, Q_bias, R_var, X, Y, idx):
    start = int(idx[seq_idx])
    xrs   = X[start:start+SL].astype(np.float64)
    yrs   = Y[start:start+SL].astype(np.float64)
    if len(xrs) < SL: return float('nan')
    v_gps = yrs * dv_iqr + dv_median
    mid   = WIN_LEN // 2
    accel_raw = xrs[:, mid, 0:3]
    quats     = xrs[:, mid, 6:10]
    accel_nav = np.zeros((SL, 3))
    for t in range(SL):
        accel_nav[t] = quat_to_rotmat(quats[t]) @ accel_raw[t] - GRAVITY_VEC
    os_ = SL // 3; oe_ = min(os_ + OE_LEN, SL)
    F_mat = np.eye(6); F_mat[0:3,3:6] = -DT*np.eye(3)
    Q = np.block([[Q_v*np.eye(3),np.zeros((3,3))],[np.zeros((3,3)),Q_bias*np.eye(3)]])
    H = np.hstack([np.eye(3),np.zeros((3,3))])
    R = R_var * np.eye(3)
    x = np.zeros(6); x[0:3] = v_gps[0]
    P = np.eye(6)*1e-2; P[3:6,3:6] = np.eye(3)*1e-3
    pos_pred = np.zeros((SL,2)); pos_true = np.zeros((SL,2))
    for t in range(1, SL):
        a_net = accel_nav[t] - x[3:6]
        x_p = np.zeros(6); x_p[0:3]=x[0:3]+a_net*DT; x_p[3:6]=x[3:6]
        P_p = F_mat@P@F_mat.T + Q
        if not (os_ <= t < oe_):
            z=v_gps[t]; S_=H@P_p@H.T+R
            K=P_p@H.T@np.linalg.solve(S_.T,np.eye(3)).T
            x=x_p+K@(z-H@x_p); P=(np.eye(6)-K@H)@P_p
        else:
            x=x_p; P=P_p
        if os_ <= t < oe_:
            if t==os_: pos_pred[t]=np.zeros(2); pos_true[t]=np.zeros(2)
            else:
                pos_pred[t]=pos_pred[t-1]+x[0:2]*DT
                pos_true[t]=pos_true[t-1]+v_gps[t,0:2]*DT
    return float(np.linalg.norm(pos_pred[oe_-1]-pos_true[oe_-1]))

def run_naive_sequence(seq_idx, X, Y, idx):
    start = int(idx[seq_idx])
    yrs   = Y[start:start+SL].astype(np.float64)
    if len(yrs)<SL: return float('nan')
    v_gps = yrs*dv_iqr+dv_median
    os_=SL//3; oe_=min(os_+OE_LEN,SL)
    v_last = v_gps[os_-1] if os_>0 else np.zeros(3)
    pp=np.zeros((SL,2)); pt=np.zeros((SL,2))
    for t in range(1,SL):
        if os_<=t<oe_:
            if t==os_: pp[t]=np.zeros(2); pt[t]=np.zeros(2)
            else: pp[t]=pp[t-1]+v_last[0:2]*DT; pt[t]=pt[t-1]+v_gps[t,0:2]*DT
    return float(np.linalg.norm(pp[oe_-1]-pt[oe_-1]))

print('EKF v2 and naive functions defined.')

In [ ]:
# ── Tune EKF on val set — never on test set ───────────────────────────────────
def ekf_objective(log_params):
    Q_v, Q_bias, R = np.exp(np.clip(log_params, -15, 5))
    drifts = [run_kf_sequence(si, Q_v, Q_bias, R, Xv, Yv, vi) for si in range(N_VAL)]
    return np.nanmean(drifts)

print('Tuning EKF on val set (Nelder-Mead, ~5 min)...')
t0 = time.time()
res_ekf = minimize(ekf_objective, np.log([Q_V_INIT, Q_B_INIT, R_INIT]),
                   method='Nelder-Mead',
                   options={'maxiter':800,'xatol':1e-3,'fatol':1e-3,'adaptive':True})
Q_V_OPT, Q_B_OPT, R_OPT = np.exp(res_ekf.x)
print(f'Done in {time.time()-t0:.0f}s  (converged={res_ekf.success})')
print(f'  Q_v={Q_V_OPT:.3e}  Q_bias={Q_B_OPT:.3e}  R={R_OPT:.3e}')
print(f'  Val mean drift (tuned): {res_ekf.fun:.3f}m')
# Save for reference
with open(os.path.join(OUT_DIR,'ekf_tuned_params.txt'),'w') as f:
    f.write(f'Q_v={Q_V_OPT:.6e}\nQ_bias={Q_B_OPT:.6e}\nR={R_OPT:.6e}\nval_mean={res_ekf.fun:.4f}\n')

In [ ]:
# ── Run all four models on a given split ──────────────────────────────────────
def run_all_models(X, Y, idx, n_seqs, split_name, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    all_res = []
    print(f'\n{"="*70}')
    print(f'  {split_name} SET — {n_seqs} sequences')
    print(f'{"="*70}')
    print(f'  {"S":>3}  {"MARSNet":>9}  {"LSTM":>8}  {"EKF":>8}  {"Naive":>8}  Group')
    print(f'  {"-"*62}')

    for si in range(n_seqs):
        start = int(idx[si])
        xrs   = X[start:start+SL];  yrs = Y[start:start+SL]
        if len(xrs) < SL: continue
        xns   = (xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq)
        os_   = SL//3; oe_ = min(os_+OE_LEN, SL)
        grp   = GROUP_MAP.get(si, '?')

        r20   = predict_sequence(model_r20,  xns, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
        rls   = predict_sequence(model_lstm, xns, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
        ekfd  = run_kf_sequence(si, Q_V_OPT, Q_B_OPT, R_OPT, X, Y, idx)
        nd    = run_naive_sequence(si, X, Y, idx)

        # Naive true position for path reference
        v_gps_phys = yrs.astype(np.float64)*dv_iqr+dv_median
        lv = v_gps_phys[os_-1] if os_>0 else np.zeros(3)
        pn = np.zeros((SL,2))
        for t in range(os_+1, SL): pn[t] = pn[t-1]+lv[:2]*DT

        all_res.append({
            'seq_idx': si, 'group': grp,
            'drift_m':       r20['drift_m'],
            'lstm_drift_m':  rls['drift_m'],
            'ekf_drift_m':   ekfd,
            'naive_drift_m': nd,
            'pos_pred':  r20['pos_pred'],
            'pos_true':  r20['pos_true'],
            'pos_gt':    np.column_stack([r20['pos_true'], np.zeros(SL)]),
            'outage_start': os_,
        })
        print(f'  {si:>3}  {r20["drift_m"]:>8.2f}m  {rls["drift_m"]:>7.2f}m  {ekfd:>7.2f}m  {nd:>7.2f}m  {grp}')

    # Summary
    m  = np.array([r['drift_m']       for r in all_res])
    ls = np.array([r['lstm_drift_m']  for r in all_res])
    ek = np.array([r['ekf_drift_m']   for r in all_res])
    na = np.array([r['naive_drift_m'] for r in all_res])

    print(f'\n  {"="*70}')
    print(f'  {"Metric":<22} {"MARSNet":>10} {"LSTM":>10} {"EKF":>10} {"Naive":>10}')
    print(f'  {"-"*65}')
    for label, fn in [
        ('Mean (m)',       lambda a: f'{a.mean():.2f}'),
        ('Median (m)',     lambda a: f'{np.median(a):.2f}'),
        ('90th pctile (m)',lambda a: f'{np.percentile(a,90):.2f}'),
        ('% under 5m',    lambda a: f'{100*np.mean(a<5):.1f}%'),
    ]:
        print(f'  {label:<22} {fn(m):>10} {fn(ls):>10} {fn(ek):>10} {fn(na):>10}')

    print(f'\n  {"Group":<20}', end='')
    for h in ['MARSNet','LSTM','EKF','Naive']: print(f' {h:>10}', end='')
    print()
    print(f'  {"-"*65}')
    for grp in GROUP_ORDER:
        gi = [i for i,r in enumerate(all_res) if r['group']==grp]
        if not gi: continue
        print(f'  {grp:<20}', end='')
        for a in [m,ls,ek,na]: print(f' {a[gi].mean():>9.2f}m', end='')
        print()
    print(f'  {"="*70}')

    # Save CSV and path arrays
    rows = [{'seq_idx':r['seq_idx'],'drift_m':r['drift_m'],
             'lstm_drift_m':r['lstm_drift_m'],'ekf_drift_m':r['ekf_drift_m'],
             'naive_drift_m':r['naive_drift_m'],'group':r['group'],
             'outage_start':r['outage_start']} for r in all_res]
    pd.DataFrame(rows).to_csv(os.path.join(save_dir,'all_results.csv'),index=False)
    # Also save MARSNet-only CSV for figures notebook (expected format)
    pd.DataFrame([{'seq_idx':r['seq_idx'],'drift_m':r['drift_m'],
                   'naive_drift_m':r['naive_drift_m'],'group':r['group'],
                   'outage_start':r['outage_start']} for r in all_res]
                ).to_csv(os.path.join(save_dir,'val_seq_results.csv'),index=False)
    # Path arrays for paper figures
    for r in all_res:
        if r['seq_idx'] in [41,47,53]:
            np.save(os.path.join(save_dir,f"path_seq{r['seq_idx']}.npy"),{
                'pos_pred':     r['pos_pred'][:,:2],
                'pos_true':     r['pos_true'][:,:2],
                'outage_start': r['outage_start'],
                'drift_m':      r['drift_m'],
                'naive_drift_m':r['naive_drift_m']})
    print(f'  Saved to {save_dir}/')
    return all_res

print('run_all_models defined.')

In [ ]:
# ── Val set — sanity check before test ────────────────────────────────────────
# MARSNet should match retrain results: mean≈6.72m, TURN≈4.73m
# LSTM should match its own notebook: mean≈11.58m (or whatever its CSV showed)
# If MARSNet mean is above 10m or TURN is above 10m: DO NOT RUN TEST SET

val_results = run_all_models(Xv, Yv, vi, N_VAL, 'VAL', VAL_DIR)

m_val = np.array([r['drift_m'] for r in val_results])
print(f'\nVal sanity check:')
print(f'  MARSNet mean: {m_val.mean():.2f}m  (expect ~6.72m from retrain)')
print(f'  If this looks right: uncomment the test cell below and run it.')
print(f'  If not: stop — check checkpoint and inference before running test set.')

In [ ]:
# ── TEST SET — run ONCE, all four models ──────────────────────────────────────
#
# Only uncomment after val sanity check passes.
# Run exactly once. These numbers go into Table 1.
# Do NOT re-run after seeing results.
#
# VAL_OK = True   # <- uncomment this line only after checking val results above
# assert VAL_OK, 'Run val sanity check first'
#
# test_results = run_all_models(Xt, Yt, ti, N_TEST, 'TEST', OUT_DIR)

print('Test cell is COMMENTED OUT — intentional.')
print('Steps:')
print('  1. Check val results above — MARSNet mean should be ~6.72m')
print('  2. Uncomment VAL_OK = True')
print('  3. Uncomment assert and test_results lines')
print('  4. Run once — save the output')